In [3]:
%load_ext autoreload
%autoreload 2
%cd /mlx_devbox/users/janne.spijkervet/repo/451/samantha
# from recipes.datasets.libritts import LibriTTSWebDataModule, SAMPLE_RATE
from recipes.datasets.librilight import LibriLightWebDataModule, SAMPLE_RATE

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
/mlx_devbox/users/janne.spijkervet/repo/451/samantha


In [4]:
import torch
from tqdm import tqdm
from webdataset import WebLoader

def test_dataloader(batch_size, shuffle_buffer_size, num_workers, max_batches):
    pl_datamodule = LibriLightWebDataModule(
        sample_rate=SAMPLE_RATE,
        batch_size=batch_size,
        shuffle_buffer_size=shuffle_buffer_size,
        num_workers=num_workers,
        resampled=True,
        shardshuffle=True,
        use_bucket_batcher=False
    )
    train_loader = pl_datamodule.train_dataloader()
    
    unique_speaker_percs = []
    speaker_ids_seen = []
    shard_seen = []
    pbar = tqdm(train_loader, total=max_batches)
    for batch_idx, batch in enumerate(pbar):
        if batch_idx == max_batches:
            break


        speaker_ids = torch.tensor(batch["speaker_id"])
        speaker_ids_seen.append(batch["speaker_id"])
        shard_seen.append(batch["shard"])
    
        unique_speaker_perc = len(speaker_ids.unique()) / len(speaker_ids)
        unique_speaker_percs.append(unique_speaker_perc)
    
        pbar.set_description(str(torch.tensor(unique_speaker_percs).mean()))

    result = {
        "batch_size": batch_size,
        "shuffle_buffer_size": shuffle_buffer_size,
        "num_workers": num_workers,
        "max_batches": max_batches,
        "speaker_ids_seen": speaker_ids_seen,
        "unique_speaker_percs": unique_speaker_percs,
        "unique_speaker_ids_perc": torch.tensor(unique_speaker_percs).mean().item(),
        "shard_seen": shard_seen,
    }
    return result

In [ ]:
result = test_dataloader(
    batch_size = 128,
    shuffle_buffer_size = 1000,
    num_workers = 4,
    max_batches = 500
)

  0%|                                                   | 0/500 [00:00<?, ?it/s]

In [ ]:
import matplotlib.pyplot as plt

all_speaker_ids = [s_ for s in result["speaker_ids_seen"] for s_ in s]
unique_speaker_ids = set(all_speaker_ids)
unique_speaker_per_batch_perc = [len(set(s)) / len(s) for s in result["speaker_ids_seen"]]
unique_shards_per_batch = [len(set(s)) for s in result["shard_seen"]]

unique_seen = set()
total_unique = []
for batch_speaker_ids in result["speaker_ids_seen"]:
    unique_seen.update(set(batch_speaker_ids))
    total_unique.append(len(unique_seen))

fig, ax = plt.subplots()
ax.plot(range(len(unique_speaker_per_batch_perc)), unique_speaker_per_batch_perc)
ax2 = ax.twinx()
ax2.plot(range(len(total_unique)), total_unique, color="red", linestyle="--")

plt.title(f"""Number of unique speaker IDs per batch\n
Unique speaker IDs seen: {len(unique_speaker_ids)}
Speaker IDs seen: {len(all_speaker_ids)}
batch_size: {result["batch_size"]}
shuffle_buffer_size: {result["shuffle_buffer_size"]}
num_workers: {result["num_workers"]}
""")
ax.set_xlabel("Number of batches")
ax.set_ylabel("Percentage of unique speaker IDs in batch")
ax2.set_ylabel("Cumulative number of unique speaker IDs seen")
plt.show()